# Gemma ince ayarı: santral anlama modeli (Gemma 3 1B / Gemma 4 E4B)

Görev: arayanın sözü → JSON (`UNDERSTAND_MODE=llm`, ADR-037). Veri `tools/egitim/anlama_veri.py` ile üretilir.

**Çalışma ortamı:** Colab → *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* (ücretsiz) ya da daha iyisi.
Model seçimi 2. adımda (`MODEL`). Gemma 3 1B çok daha küçüktür; E4B QLoRA ~10 GB VRAM ister, T4'ün 15 GB'ı ikisine de yeter.

Sırayla çalıştırın. Her adımın sonunda **kontrol** satırları var: beklenen çıktıyı vermezse sonraki adıma geçmeyin.

In [ ]:
%%capture
!pip install --upgrade unsloth

## 1. Veri
`anlama_egitim.jsonl` ve `anlama_dogrulama.jsonl` dosyalarını yükleyin (bilgisayarınızda
`python tools/egitim/anlama_veri.py` çalıştırınca `tools/egitim/veri/` altında oluşur).

In [ ]:
import json
import os

if not os.path.exists("anlama_egitim.jsonl"):
    from google.colab import files
    files.upload()          # iki dosyayı birlikte seçin

def oku(yol):
    with open(yol, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

egitim, dogrulama = oku("anlama_egitim.jsonl"), oku("anlama_dogrulama.jsonl")
print(len(egitim), "eğitim,", len(dogrulama), "doğrulama")
print(egitim[0]["messages"][1]["content"], "=>", egitim[0]["messages"][2]["content"])

## 2. Model (4-bit, QLoRA)
`MODEL` ile seçin. Şablon ve tur işaretleri model ailesine göre kendiliğinden ayarlanır.

In [ ]:
import torch
from unsloth import FastModel

MODEL = "gemma-3-1b"    # "gemma-3-1b" (google/gemma-3-1b-it) ya da "gemma-4-e4b"
MAX_UZUNLUK = 3072      # istem + birim listesi en uzun ~1.500 token; kontrol aşağıda

# Hız (v9, 26.11.2026): v8 A100'de 3,5 saat sürdü (4 bit + gradyan CPU'ya taşıma + 4'lük parti). Büyük GPU'da
# (A100 / L4, >= 20 GB) 1B model 16 bit LoRA ile, taşımasız ve 16'lık partiyle eğitilir: aynı etkin parti (16),
# birkaç kat hızlı. T4'te (16 GB) eski ayar kalır.
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
HIZLI = GPU_GB >= 20 and MODEL == "gemma-3-1b"
print(f"GPU {torch.cuda.get_device_name(0)} {GPU_GB:.0f} GB → {'hızlı ayar' if HIZLI else 'T4 ayarı'}")

AYAR = {
    "gemma-3-1b":  dict(ad="unsloth/gemma-3-1b-it",  sablon="gemma-3",
                        kullanici="<start_of_turn>user\n", model_turu="<start_of_turn>model\n", ollama="gemma3:1b"),
    "gemma-4-e4b": dict(ad="unsloth/gemma-4-E4B-it", sablon="gemma-4",
                        kullanici="<|turn>user\n", model_turu="<|turn>model\n", ollama="gemma4:e4b"),
}[MODEL]

model, tokenizer = FastModel.from_pretrained(
    model_name = AYAR["ad"],
    max_seq_length = MAX_UZUNLUK,
    load_in_4bit = not HIZLI,                # 1B model 16 bit ~2 GB: büyük GPU'da sıkıştırmaya gerek yok
    full_finetuning = False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,    # yalnız metin (Gemma 3 1B zaten yalnız metin)
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    random_state = 3407,
    use_gradient_checkpointing = False if HIZLI else "unsloth",   # "unsloth": gradyanları CPU'ya taşır (yavaş)
)

## 3. Sohbet şablonu
Üretimde düşünme kapalı (`think: false`); eğitim ve kullanım aynı biçimde olmalı.

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = AYAR["sablon"])

def bicimle(ornek):
    try:
        metin = tokenizer.apply_chat_template(ornek["messages"], tokenize=False, add_generation_prompt=False,
                                              enable_thinking=False)
    except TypeError:
        metin = tokenizer.apply_chat_template(ornek["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": metin.removeprefix("<bos>")}


ds_egitim = Dataset.from_list([bicimle(o) for o in egitim])
ds_dogrulama = Dataset.from_list([bicimle(o) for o in dogrulama])

print(ds_egitim[0]["text"][-400:])
uzunluk = max(len(tokenizer(t)["input_ids"]) for t in ds_egitim["text"])
print("\nen uzun:", uzunluk, "token")
# KONTROL: kullanıcı ve model turları görünmeli, JSON model turunda olmalı. (Gemma 3'te ayrı sistem turu yoktur:
# şablon sistem istemini ilk kullanıcı turunun başına koyar; Ollama da aynısını yapar.)
assert AYAR["model_turu"] in ds_egitim[0]["text"], "şablon beklenen biçimde değil"
assert uzunluk < MAX_UZUNLUK

## 4. Eğitim
Yalnız modelin cevabı (JSON) öğrenilir; sistem istemi ve söz maskelenir.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds_egitim,
    eval_dataset = ds_dogrulama,
    args = SFTConfig(
        dataset_text_field = "text",
        packing = False,                      # yeni Unsloth/TRL paketliyordu (25.11.2026: 9458 örnek → 2563 blok,
                                              # 13 saat; yalnız cevabın öğrenilmesi bozulur). Kontrol: Num examples
        padding_free = False,                 # yeni Unsloth bunu da kendiliğinden açabiliyor
        max_seq_length = MAX_UZUNLUK,
        per_device_train_batch_size = 16 if HIZLI else 4,
        gradient_accumulation_steps = 1 if HIZLI else 4,      # etkin parti her iki ayarda 16
        per_device_eval_batch_size = 16 if HIZLI else 4,
        group_by_length = True,               # benzer uzunluklar aynı partide: dolgu (padding) israfı azalır
        bf16 = torch.cuda.is_bf16_supported(), fp16 = not torch.cuda.is_bf16_supported(),
        dataloader_num_workers = 2,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        warmup_steps = 10,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        logging_steps = 10,
        eval_strategy = "steps", eval_steps = 200,     # v8: 100 (doğrulama da süre alıyordu)
        save_strategy = "steps", save_steps = 200, save_total_limit = 2,
        output_dir = "cikti",
        seed = 3407,
        report_to = "none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part = AYAR["kullanici"],
                                 response_part = AYAR["model_turu"])

# KONTROL: öğrenilen kısım yalnız JSON olmalı (maskelenmemiş etiketler)
lbl = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
print("öğrenilen:", tokenizer.decode(lbl))
print("eğitim örneği:", len(trainer.train_dataset), "(anlama_egitim.jsonl satır sayısıyla aynı olmalı)")
assert len(trainer.train_dataset) == len(egitim), "örnekler paketlenmiş: packing / padding_free kapalı mı?"


In [ ]:
istatistik = trainer.train()
print(istatistik)
# Not (Unsloth): Gemma 4 E4B'de ilk kayıp 13–15 civarı normaldir (çok kipli model); Gemma 3 1B'de daha düşük
# başlar. 100+ ise bir şey yanlış (gradyan biriktirme).

## 5. Hızlı deneme (doğrulama setinden)

In [ ]:
import random

FastModel.for_inference(model)

dogru = 0
for o in random.Random(0).sample(dogrulama, 20):
    msgs = o["messages"][:2]
    try:
        giris = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                              return_dict=True, enable_thinking=False).to("cuda")
    except TypeError:
        giris = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                              return_dict=True).to("cuda")
    cikti = model.generate(**giris, max_new_tokens=80, do_sample=False)
    metin = tokenizer.decode(cikti[0][giris["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    beklenen = o["messages"][2]["content"]
    dogru += metin == beklenen
    print(("✓" if metin == beklenen else "✗"), msgs[1]["content"], "→", metin)
print(f"\n{dogru}/20 birebir aynı")

## 6. Kaydet ve indir
- `gemma-santral-lora/`: LoRA adaptörü (sonra yeniden eğitmek için)
- `.gguf`: Ollama'ya yüklenecek model (1B: Q8_0 ~1 GB; E4B: Q4_K_M ~5 GB)

In [ ]:
# 1B küçük: Q8_0 (~1 GB) doğruluğu korur; E4B için q4_k_m (~5 GB) yeterli
import glob
import os

SURUM = "v9"     # ← her eğitimde değiştirin: Drive klasörü ve dosya adları buna göre (önceki sürümün üstüne yazılmaz)
NICEMLEME = "q8_0" if MODEL == "gemma-3-1b" else "q4_k_m"
model.save_pretrained("gemma-santral-lora")
tokenizer.save_pretrained("gemma-santral-lora")
model.save_pretrained_gguf("gemma-santral", tokenizer, quantization_method = NICEMLEME)
# f16 GGUF da saklanır (~2 GB): ileride q4_K_M gibi başka bir nicemleme bundan üretilir. q8'den yeniden
# nicemlemek kalite kaybettirir (20.11.2026: q8 → q4 iki diyalog senaryosunu düşürdü).
model.save_pretrained_gguf("gemma-santral-f16", tokenizer, quantization_method = "f16")

# Unsloth dosyaya taban modelin adını veriyor ("gemma-3-1b-it.Q8_0.gguf", 25.11.2026): sürümlü ada çevrilir
for yol in glob.glob("gemma-santral*/*.gguf") + glob.glob("*.gguf"):
    tur = "F16" if "F16" in yol.upper() or "-f16" in yol else NICEMLEME.upper()
    yeni = f"gemma-santral-{SURUM}.{tur}.gguf"
    if os.path.basename(yol) != yeni:
        os.replace(yol, yeni)
print(sorted(glob.glob("gemma-santral-*.gguf")))


In [ ]:
# İndirme: büyük dosyada tarayıcı indirmesi yavaş olabilir; Google Drive'a kopyalamak daha güvenli.
# Klasör sürümlüdür (MyDrive/gemma-santral-v9): önceki sürüm korunur.
from google.colab import drive

drive.mount("/content/drive")
HEDEF = f"/content/drive/MyDrive/gemma-santral-{SURUM}"
!mkdir -p "{HEDEF}"
!cp -r gemma-santral-lora "{HEDEF}/"
!cp gemma-santral-*.gguf "{HEDEF}/"
!ls -la "{HEDEF}"
print(f"İndirilecek dosya: {HEDEF}/gemma-santral-{SURUM}.{NICEMLEME.upper()}.gguf")


## 7. Sonra (kendi makinenizde)
```bash
bash tools/egitim/ollama_yukle.sh D:/modeller/gemma-santral-v9.Q8_0.gguf gemma-santral-v9 gemma3:1b
python tools/egitim/anlama_olc.py --model gemma-santral --model gemma-santral-v9 -v   # kurulumdaki ile yeni
```